# 检索、引用与模型回答：逐层检查

先看真实模型结果，再运行机制实验。下面读取的是已经单独运行并固定版本的模型报告；本 Notebook 不会把报告读取当成重新调用模型。完整重跑命令见 README。

In [1]:
from pathlib import Path
import sys, json, tempfile
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/run_python.py").exists())
for p in ROOT.rglob("src"):
    if not {"node_modules", ".git", ".venv"}.intersection(p.parts):
        sys.path.insert(0, str(p))


In [2]:
base=ROOT/'20-Projects/learning-workbench/artifacts/real-models'
retrieval=json.loads((base/'retrieval.json').read_text())
print(json.dumps(retrieval,ensure_ascii=False,indent=2)[:16000])

{
  "models": [
    {
      "model": "sentence-transformers/paraphrase-MiniLM-L3-v2",
      "revision": "4ca70771034acceecb2e72475f72050fcdde4ddc"
    },
    {
      "model": "cross-encoder/ms-marco-MiniLM-L6-v2",
      "revision": "233902d25c440f23af6f7d6e94d2946bac0bee0a"
    }
  ],
  "trials": [
    {
      "mode": "bm25",
      "id": "q1",
      "query": "What should I do before opening the fan?",
      "retrieved": [
        "fan-v1",
        "fan-v2",
        "network"
      ],
      "relevant": [
        "fan-v1",
        "fan-v2"
      ],
      "recall_at_3": 1.0,
      "reciprocal_rank": 1.0,
      "empty_returned": false
    },
    {
      "mode": "bm25",
      "id": "q2",
      "query": "How can I recover an erased document?",
      "retrieved": [
        "backup"
      ],
      "relevant": [
        "backup"
      ],
      "recall_at_3": 1.0,
      "reciprocal_rank": 1.0,
      "empty_returned": false
    },
    {
      "mode": "bm25",
      "id": "q3",
      "query": "The 

## 引用存在，不等于主张成立

查看 g2：引文谈温度，主张却谈断电。请分别检查引用是否存在、是否支持这条主张、是否回答了题目。

In [3]:
generation=json.loads((base/'generation.json').read_text())
review=json.loads((base/'generation-review.json').read_text())
print(json.dumps({'generation':generation,'independent_review':review},ensure_ascii=False,indent=2))

{
  "generation": {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "revision": "7ae557604adf67be50417f59c2c2f167def9a775",
    "trials": [
      {
        "id": "g1",
        "error_type": "ProviderError",
        "error": "model output is not a complete JSON object",
        "abstention_correct": false
      },
      {
        "id": "g2",
        "question": "FAN-01: What is the DEMO-A temperature limit?",
        "answer": {
          "abstained": false,
          "reason": "The text directly states the temperature limit for FAN-01 as 68°C.",
          "claims": [
            {
              "text": "Before inspecting a cooling fan, disconnect power",
              "source_id": "fan-v2@2:0-114:f33f91e9bd85",
              "quote": "Version 2 changes the DEMO-A temperature limit to 68 C."
            },
            {
              "text": "Before inspecting a cooling fan, disconnect power",
              "source_id": "fan-v2@2:0-114:f33f91e9bd85",
              "quote": "Version 2 cha

## 实际重跑记忆任务

同一任务保留无记忆与有记忆输出。观察一次性要求为什么不应写入长期记忆。

In [4]:
from learning_workbench.cli import read_jsonl
from learning_workbench.memory import evaluate_memory
with tempfile.TemporaryDirectory() as directory:
    memory=evaluate_memory(read_jsonl('memory-tasks.jsonl'),directory)
assert all(row['correct'] for row in memory)
print([(row['id'],row['baseline_correct'],row['correct']) for row in memory])

[('m0', False, True), ('m1', True, True), ('m2', True, True), ('m3', True, True), ('m4', True, True), ('m5', True, True), ('m6', True, True), ('m7', False, True)]


## 以 Task 为抽样单位

这里是构造数据上的统计演示。比较配对 Task bootstrap 与错误地打散 Trial 的区间宽度；不要把区间解释为真实模型收益。

In [5]:
from learning_workbench.experiments import statistics_experiment
print(json.dumps(statistics_experiment()['report'],indent=2))

{
  "sampling_unit": "Task",
  "task_count": 12,
  "trial_counts": [
    5,
    5,
    5,
    5,
    5,
    5,
    5,
    5,
    5,
    5,
    5,
    5
  ],
  "mean_paired_difference": 0.0625,
  "task_bootstrap_95": [
    -0.05000000000000001,
    0.17500000000000002
  ],
  "naive_trial_bootstrap_95": [
    0.0024999999999999983,
    0.1225
  ],
  "seed": 7,
  "repeats": 1000
}
